# ESM-2

In [ ]:
!pip install fair-esm==2.0.0

In [ ]:
# สำหรับ ESM < 1028 เปิด GPU
import esm
import torch
import collections
import pandas as pd
import gc

def esm_embeddings(peptide_sequence_list, model_name, batch_size=1):
    model_dict = {
        'esm2_t6_8M_UR50D': (esm.pretrained.esm2_t6_8M_UR50D, 6),
        
    }
    
    if model_name not in model_dict:
        raise ValueError(f"Invalid model name '{model_name}'. Please choose from {list(model_dict.keys())}.")
    
    model_func, num_layers = model_dict[model_name]
    model, alphabet = model_func()
    
    batch_converter = alphabet.get_batch_converter()
    model.eval()  
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Enable mixed precision
    scaler = torch.cuda.amp.GradScaler()
    embeddings_results = []

    # Process sequences in batches
    for i in range(0, len(peptide_sequence_list), batch_size):
        batch_sequences = peptide_sequence_list[i:i + batch_size]
        batch_labels, batch_strs, batch_tokens = batch_converter(batch_sequences)
        batch_tokens = batch_tokens.to(device)
        
        with torch.cuda.amp.autocast():
            with torch.no_grad():
                results = model(batch_tokens, repr_layers=[num_layers])  # Disable contacts to save memory

        token_representations = results["representations"][num_layers] 
        batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

        for j, tokens_len in enumerate(batch_lens):
            sequence_representation = token_representations[j, 1 : tokens_len - 1].mean(0)
            embeddings_results.append(sequence_representation.cpu().numpy())

        # Clear memory
        del batch_tokens, results, token_representations
        torch.cuda.empty_cache()
        gc.collect()

    # Convert embeddings to DataFrame
    embeddings_results = pd.DataFrame(embeddings_results)
    return embeddings_results

# Load the dataset
dataset = pd.read_csv('/kaggle/input/mapms-raw/alt_test.csv', na_filter=False) 
#column_names = ['Column1', 'seq'] 
#dataset = pd.read_csv('/kaggle/input/clathrin-2-0-6/Dat06_forPSSM.csv', header=None, names=column_names, na_filter=False)


sequence_list = dataset['seq']
peptide_sequence_list = []

# Prepare sequence_list for ESM processing
for seq in sequence_list:
    format_seq = [seq, seq]  # Input format setting in ESM model, [name, sequence]
    peptide_sequence_list.append(tuple(format_seq))


embeddings_results = esm_embeddings(peptide_sequence_list,'esm2_t6_8M_UR50D')
embeddings_results = pd.DataFrame(embeddings_results)
embeddings_results.columns = [f"ESM320_{i}" for i in range(embeddings_results.shape[1])]
embeddings_results.to_csv('ESM-alt_test.csv', index=True)



# TPC

In [ ]:
import pandas as pd
import itertools

# -----------------------------
# 1) อ่านไฟล์ CSV
# -----------------------------
df = pd.read_csv("/kaggle/input/mapms-raw/alt_test.csv")   # ต้องมีคอลัมน์ชื่อ 'seq'

# -----------------------------
# 2) เตรียม tri-peptide ทั้งหมด
# -----------------------------
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
tri_peptides = [''.join(p) for p in itertools.product(amino_acids, repeat=3)]

# -----------------------------
# 3) ฟังก์ชันคำนวณ TPC
# -----------------------------
def calculate_tpc(sequence):
    counts = dict.fromkeys(tri_peptides, 0)
    total = len(sequence) - 2

    if total <= 0:
        return counts

    for i in range(total):
        tri = sequence[i:i+3]
        if tri in counts:
            counts[tri] += 1

    # normalize
    for k in counts:
        counts[k] /= total

    return counts

# -----------------------------
# 4) แปลง sequence → TPC feature
# -----------------------------
tpc_df = df["seq"].apply(lambda x: pd.Series(calculate_tpc(x)))

# รวมกับ dataframe เดิม
df_tpc = pd.concat([df, tpc_df], axis=1)

# -----------------------------
# 5) บันทึกผลลัพธ์
# -----------------------------
df_tpc.to_csv("tpc-alt_test.csv", index=True)

print("TPC feature extraction complete!")


# Mol2Vec

In [ ]:
!pip install rdkit-pypi
!pip install pandas==0.23.0
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

!pip install git+https://github.com/samoturk/mol2vec;

In [ ]:
import pandas as pd
# นำเข้าข้อมูล
mdf = pd.read_csv('/kaggle/input/mapms-smiles-selfies/smiles_main_train.csv')
#mdf = mdf[mdf['Class'] != 'intermediate']
#mdf =mdf.loc[0:3]
#smiles_name = mdf['Name'].values
smiles = mdf['smiles'].values

# Define y
#y_all = mdf['cls'].values
#y_df = pd.DataFrame(y_all)

In [ ]:
import numpy as np
import pandas as pd
from mol2vec.features import mol2alt_sentence, MolSentence, sentences2vec
from gensim.models import Word2Vec
from rdkit import Chem  # Import the Chem module

# Assuming 'model' is the 100-dimensional Mol2Vec model (make sure you're using the 100-dimensional model)
model = Word2Vec.load("/kaggle/input/mol2vec100d/model_100dim.pkl")
mdf['mol'] = mdf['smiles'].apply(lambda x: Chem.MolFromSmiles(x))

# Replace 'model.wv.vocab' with 'model.wv.key_to_index'
def sentences2vec_updated(sentences, model, unseen='UNK'):
    keys = set(model.wv.key_to_index.keys())
    vec = []
    for word in sentences:
        if word in keys:
            vec.append(model.wv[word])
        else:
            vec.append(model.wv[unseen])
    return np.array(vec)  # Convert the list to a numpy array

# Function to apply Mol2Vec to the entire DataFrame and save to CSV
def save_mol2vec_features(mdf, model, output_csv='mol2vec_features.csv'):
    features = []
    indices = []  # List to store original indices
    
    for idx, row in mdf.iterrows():
        mol = row['mol']
        
        # Check if the molecule is valid
        if mol is None:
            print(f"Skipping invalid SMILES at index {idx}")
            continue
        
        # Convert SMILES to Mol object
        try:
            mol_sentence = mol2alt_sentence(mol, radius=1)
            sentence_obj = MolSentence(mol_sentence)
        
            # Get the Mol2Vec feature vector for the sentence
            vector = sentences2vec_updated(sentence_obj, model, unseen='UNK')
        
            # No need to truncate or pad if you're using the 100-dimensional Mol2Vec model
            # Perform averaging across the word vectors to create a single vector
            vector = np.mean(vector, axis=0)
        
            # Flatten the vector into a 1D array
            features.append(vector.flatten())  # Flatten to a 1D array for each molecule
            indices.append(idx)  # Store the original index
        except Exception as e:
            print(f"Error processing molecule at index {idx}: {e}")
            continue
    
    # Convert list of feature vectors into a DataFrame
    feature_df = pd.DataFrame(features)
    
    # Add the original indices as a new column to the DataFrame
    feature_df['index'] = indices
    
    # Set the index to be the original indices
    feature_df.set_index('index', inplace=True)
    
    # Save the DataFrame to a CSV file, keeping the original indices
    feature_df.to_csv(output_csv, index=True)
    print(f"Mol2Vec features saved to {output_csv}")

# Assuming 'mdf' is your DataFrame and it has a 'mol' column with SMILES strings
# Apply the function to save features to CSV
save_mol2vec_features(mdf, model, output_csv='R1D100_main_train.csv')  # Using the 100-dimensional model
